check answer extraction functions written in 

/n/home07/than157/desktop/done-large_projects/learn-better/evolm/evaluation/cot-eval-harness/cot_eval/evaluation/Evaluator.py

for medmcqa

### can use analy environment -- doesn't really matter, only uses basic python functions

In [1]:
import re

In [2]:
#check PubMedQAEvaluator's _extract_answer_from_gold_solution(self, solution: str)
solution = "The conclusion of the abstract is: Using the charts described, there was only a slight overestimation of visual acuity by the Snellen E compared to the Landolt C, even in strabismus amblyopia. Small differences in the lower visual acuity range have to be considered. Therefore, the answer is no."
solution = solution.split("Therefore, the answer is ")[-1].rstrip('.')
solution

'no'

### check PubMedQAEvaluator's _extract_answer_from_model_completion

In [3]:
#from Evaluator._extract_answer_from_model_completion -- super() function

def last_boxed_only_string(text: str) -> str | None:
    # Source: https://github.com/huggingface/lighteval/blob/d7a1f1128deb8d76d36650339796c81521b61958/src/lighteval/metrics/normalizations.py#L122
    """Extract the last \\boxed{...} or \\fbox{...} element from a string."""

    idx = text.rfind("\\boxed")
    if idx < 0:
        idx = text.rfind("\\fbox")
        if idx < 0:
            return None

    i = idx
    right_brace_idx = None
    num_left_braces_open = 0
    while i < len(text):
        if text[i] == "{":
            num_left_braces_open += 1
        if text[i] == "}":
            num_left_braces_open -= 1
            if num_left_braces_open == 0:
                right_brace_idx = i
                break
        i += 1

    if right_brace_idx is None:
        retval = None
    else:
        retval = text[idx : right_brace_idx + 1]

    return retval


def remove_boxed(text: str | None) -> str:
    # Source: https://github.com/huggingface/lighteval/blob/d7a1f1128deb8d76d36650339796c81521b61958/src/lighteval/metrics/normalizations.py#L98
    """
    Extract the text within a \\boxed{...} environment.
    Example:
    >>> remove_boxed(\\boxed{\\frac{2}{3}})
    \\frac{2}{3}
    """
    if text is None:
        return ""
    try:
        if "\\boxed " in text:
            left = "\\boxed "
            assert text[: len(left)] == left
            return text[len(left) :]

        left = "\\boxed{"

        assert text[: len(left)] == left
        assert text[-1] == "}"

        return text[len(left) : -1]
    except Exception:
        return ""




def super_extract_answer_from_model_completion(completion, answer_extraction_format):
    def extract_from_boxed(c):
        box = last_boxed_only_string(c)
        ans = remove_boxed(box)
        return ans if ans else None

    def extract_from_answer_is(c):
        split = c.split("answer is")
        if len(split) > 1:
            answer = split[-1].strip()
            if ":" in answer:
                answer = answer.split(":")[-1].strip()
            return answer
        else:
            return None

    if answer_extraction_format == "boxed":
        return extract_from_boxed(completion)
    elif answer_extraction_format == "answer is":
        return extract_from_answer_is(completion)
    elif answer_extraction_format == "both":
        boxed_ans = extract_from_boxed(completion)
        if boxed_ans:
            return boxed_ans

        answer_is_ans = extract_from_answer_is(completion)
        if answer_is_ans:
            return answer_is_ans

        return None
    else:
        raise ValueError(f"Invalid answer extraction format {answer_extraction_format}")



In [4]:
#from PubMedQAEvaluator's ._extract_answer_from_model_completion() function


def _extract_answer_from_model_completion(completion: str) -> str | None:
        ans = super_extract_answer_from_model_completion(completion, 'both')
        print('answer after super fn: ', ans)
        ### case 1: if answer is None, return None
        if not ans:
            return None

        ### case 2: if answer is a string, return the first word in the answer
             # first word should be 'yes', 'maybe', or 'no'
             # return the first word in the answer whatever it is
        ans = ans.strip()
        ans = ans.split()[0] #split into individual words, get first word
        ans = ans.rstrip(".,;!?") #remove trailing punctuation from answer
        return ans
        

In [5]:
#example completions
completion = "The conclusion of the abstract is: Using the charts described, there was only a slight overestimation of visual acuity by the Snellen E compared to the Landolt C, even in strabismus amblyopia. Small differences in the lower visual acuity range have to be considered. Therefore, the answer is no. More text to follow."

example_completions = [
    completion,
    "Text at the start. Therefore, the answer is no.",
    "Text at the start. Therefore, the answer is no. More text to follow.",
    "Text at the start. Therefore, the answer is no, but i'm not sure. More text to follow.",
    ]



In [6]:
for c in example_completions:
    answer_after_pubmedqa_fn = _extract_answer_from_model_completion(c)
    print('answer_after_pubmedqa_fn: ', answer_after_pubmedqa_fn)
    print('')

answer after super fn:  no. More text to follow.
answer_after_pubmedqa_fn:  no

answer after super fn:  no.
answer_after_pubmedqa_fn:  no

answer after super fn:  no. More text to follow.
answer_after_pubmedqa_fn:  no

answer after super fn:  no, but i'm not sure. More text to follow.
answer_after_pubmedqa_fn:  no

